<h2>Upload video to youtube</h2>

In [17]:
import os

from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import random

import shutil
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'core')))   #import sibling files

import importlib
import utilities

importlib.reload(utilities)
from utilities import get_first_file, rename_files

import argparse
import json

from dotenv import load_dotenv
load_dotenv(override=True)


True

In [18]:
PROJECT_FOLDER = "/Users/sangdo/Documents/Source/Python/python_social_content/"
PROJECT_CORE_FOLDER = f"{PROJECT_FOLDER}core/"

YT_CREDENTIAL_FILEPATH = PROJECT_CORE_FOLDER + "secret_files/martin_yt_client_secret.json" #download from Google console app
YT_SECRET_FILEPATH = PROJECT_CORE_FOLDER + 'secret_files/martin_yt_token.json' #appear after authentication in web (Note this is for 1 channel only)

In [19]:
SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload",   #upload file
    "https://www.googleapis.com/auth/youtube.force-ssl" #create comment
]


In [20]:
DATA_FOLDER = '/Users/sangdo/Downloads/math_games_video/'
VIDEO_FOLDER = f'{DATA_FOLDER}output/'   #contain mp4 files

In [21]:
COMMENT = os.getenv('COMMENT_CONTENT').replace('\\n', '\n')

In [22]:
#authorize to save permanent token
def get_authenticated_service():
    creds = None
    # load saved token
    if os.path.exists(YT_SECRET_FILEPATH):
        print("Using saved Youtube token")
        creds = Credentials.from_authorized_user_file(YT_SECRET_FILEPATH, SCOPES)

    # if no valid credentials
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                YT_CREDENTIAL_FILEPATH,
                SCOPES
            )
            flow.oauth2session.params["access_type"] = "offline"
            creds = flow.run_local_server(port=0)
        # save token again
        with open(YT_SECRET_FILEPATH, "w") as token:
            token.write(creds.to_json())

    youtube = build("youtube", "v3", credentials=creds)
    return youtube

youtube = get_authenticated_service() #permanent token is saved. Note: login with developer account (Tester type)

Using saved Youtube token


In [23]:
def upload_video(file_path, title, description):
    request_body = {
        "snippet": {
            "title": title,
            "description": description
        },
        "status": {
            "privacyStatus": "public"
        }
    }

    media = MediaFileUpload(file_path, chunksize=8*1024*1024, resumable=True)

    request = youtube.videos().insert(
        part="snippet,status",
        body=request_body,
        media_body=media
    )

    response = None

    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"Uploading {int(status.progress() * 100)}%")

    print("Upload complete")
    print("Video ID:", response["id"])
    return response["id"]

In [24]:
def get_random_item(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if not data:
        raise ValueError("JSON file is empty")
    
    return random.choice(data)

# Usage
# item = get_random_item(TITLE_FILEPATH)
# print(item['title'])
# print(item['description'])

In [25]:
def auto_upload_video():
    first_mp4_file, filename = get_first_file(VIDEO_FOLDER, 'mp4')  #order by index
    item = get_random_item(TITLE_FILEPATH)  #get random titles and description
    print('Begin uploading: ' + first_mp4_file)
    new_video_id = upload_video(first_mp4_file, item['title'], item['description'])
    print('===== Done uploading: ' + first_mp4_file)
    #rename the file
    os.rename(first_mp4_file, VIDEO_FOLDER + new_video_id + ".cmt")   #need to comment in this video

In [26]:
def add_comment(video_id, comment_text):
    request = youtube.commentThreads().insert(
        part="snippet",
        body={
            "snippet": {
                "videoId": video_id,
                "topLevelComment": {
                    "snippet": {
                        "textOriginal": comment_text
                    }
                }
            }
        }
    )

    response = request.execute()
    print("Comment posted")
    return response
#
def auto_add_comment():
    #find the first file in path
    video_path, video_id = get_first_file(VIDEO_FOLDER, 'cmt')
    add_comment(video_id, COMMENT)
    os.rename(video_path, VIDEO_FOLDER + video_id + ".fb")   #need to post to FB this video

In [27]:
# rename_files(VIDEO_FOLDER, 'mp4', 13)
# rename_files('/Users/sangdo/Downloads/math_games_video/img/', 'png', 1)

<h2>Read the json file that contains titles</h2>

In [28]:
TITLE_FILEPATH = f'{DATA_FOLDER}math_titles_500.json'

def get_all_titles(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return [item['title'] for item in data]

# Usage
# titles = get_all_titles(TITLE_FILEPATH)
# for title in titles:
#     print(title)

In [29]:
def remove_duplicate_titles(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    seen = set()
    unique_data = []
    for item in data:
        if item['title'] not in seen:
            seen.add(item['title'])
            unique_data.append(item)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(unique_data, f, ensure_ascii=False, indent=2)

    print(f"Before: {len(data)} items, After: {len(unique_data)} items, Removed: {len(data) - len(unique_data)}")

# Usage
# remove_duplicate_titles(TITLE_FILEPATH)

In [30]:
#testing in jupyter
# sys.argv = ['script.py', '-type', 'comment']
#load parameters
parser = argparse.ArgumentParser()
parser.add_argument('-type', type=str, default=None, help='Action type: upload_video or comment')

args,_ = parser.parse_known_args()
if args.type is None:
    print('Type not found')
    raise SystemExit
action_type = args.type
print('action type:', action_type)
if action_type == 'upload_video':
    auto_upload_video()
elif action_type == 'comment':
    auto_add_comment()


Type not found


SystemExit: 